# Week 1 — LLM Fundamentals & Environment Setup

**AI Agentic Engineering · Corte 1**

Companion notebook to `week-01-llm-fundamentals-content.html`. Run the cells in order during lab time.

**You will practice:**
1. Making your first Gemini API call from Python.
2. Seeing `temperature`, `top_p`, and `top_k` change model output on the *same* prompt.
3. Counting tokens for real and comparing it to the "1 token ≈ 4 characters" rule of thumb.
4. Making the *same* call through three tools: the raw SDK, a minimal Google ADK agent, and a minimal LangChain chat model.
5. Two open exercises.

**Before you start:** create a free API key at [Google AI Studio](https://aistudio.google.com/app/apikey) and save it in a `.env` file next to this notebook:

```
GOOGLE_API_KEY="paste-your-key-here"
```


In [1]:
%pip install -q --upgrade google-genai google-adk langchain-google-genai langgraph python-dotenv


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 1. Environment check

Load the API key and make one call to confirm everything is wired up.

In [2]:
import time
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in a .env file first."

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
MODEL = "gemini-flash-latest"

response = client.models.generate_content(
    model=MODEL,
    contents="In one sentence, what is a large language model?",
)
print(((response.text if getattr(response, 'text', None) else 'NO TEXT GENERATED') if (response.text if getattr(response, 'text', None) else 'NO TEXT GENERATED') else 'NO TEXT GENERATED'))
time.sleep(4)


AssertionError: Set GOOGLE_API_KEY in a .env file first.

## 2. Sampling parameters: temperature, top-p, top-k

We'll send the **same prompt** several times, changing only `temperature`. Watch how the output goes from
predictable to varied.

In [3]:
import time
PROMPT = "Give me one creative name for a coffee shop. Reply with just the name."

for temp in [0.0, 0.4, 0.9, 1.5]:
    response = client.models.generate_content(
        model=MODEL,
        contents=PROMPT,
        config=types.GenerateContentConfig(temperature=temp, max_output_tokens=20),
    )
    print(f"temperature={temp:<4} -> {((response.text if getattr(response, 'text', None) else 'NO TEXT GENERATED').strip() if (response.text if getattr(response, 'text', None) else 'NO TEXT GENERATED') else 'NO TEXT GENERATED')}")
time.sleep(4)


NameError: name 'client' is not defined

Run the cell above a few times. At `temperature=0.0` the answer should barely change between runs.
At `temperature=1.5` you should see real variety (and occasionally something a little unhinged — that's expected).

Now let's isolate `top_p` and `top_k` by holding temperature fixed at a mid-range value.

In [4]:
import time
for top_p in [0.1, 0.5, 0.95]:
    response = client.models.generate_content(
        model=MODEL,
        contents=PROMPT,
        config=types.GenerateContentConfig(temperature=0.9, top_p=top_p, max_output_tokens=20),
    )
    print(f"top_p={top_p:<5} -> {((response.text if getattr(response, 'text', None) else 'NO TEXT GENERATED').strip() if (response.text if getattr(response, 'text', None) else 'NO TEXT GENERATED') else 'NO TEXT GENERATED')}")

print()
for top_k in [1, 5, 40]:
    response = client.models.generate_content(
        model=MODEL,
        contents=PROMPT,
        config=types.GenerateContentConfig(temperature=0.9, top_k=top_k, max_output_tokens=20),
    )
    print(f"top_k={top_k:<3} -> {((response.text if getattr(response, 'text', None) else 'NO TEXT GENERATED').strip() if (response.text if getattr(response, 'text', None) else 'NO TEXT GENERATED') else 'NO TEXT GENERATED')}")
time.sleep(4)


NameError: name 'client' is not defined

## 3. Tokens: count them for real

The rule of thumb is "1 token ≈ 4 characters ≈ 0.75 words" for English. Let's check it against the real
tokenizer using `count_tokens`.

In [5]:
import time
samples = [
    "Hi!",
    "The capital of France is Paris.",
    "AI Agentic Engineering is a course about building autonomous systems with large language models, "
    "retrieval-augmented generation, and multi-agent orchestration frameworks like Google ADK and LangGraph.",
]

for text in samples:
    result = client.models.count_tokens(model=MODEL, contents=text)
    chars = len(text)
    tokens = result.total_tokens
    ratio = chars / tokens if tokens else 0
    print(f"chars={chars:<4} tokens={tokens:<4} chars/token={ratio:.2f}  | {text[:50]!r}")
time.sleep(4)


NameError: name 'client' is not defined

## 4. Same call, three tools

Below, the identical prompt is sent through: **(A)** the raw `google-genai` SDK, **(B)** a one-line **Google ADK**
agent run with `InMemoryRunner`, and **(C)** a **LangChain** chat model. This is a *light preview* — we are not
building real agents yet (tools, multi-step reasoning, ReAct) until Corte 2, Week 6 onward. The goal today is
just to recognize the same underlying API call under each interface.

In [6]:
import time
# --- A. Raw SDK ---
prompt = "Name one famous mathematician and the theorem they're best known for."

response = client.models.generate_content(model=MODEL, contents=prompt)
print("A) raw SDK:\n", ((response.text if getattr(response, 'text', None) else 'NO TEXT GENERATED') if (response.text if getattr(response, 'text', None) else 'NO TEXT GENERATED') else 'NO TEXT GENERATED'))
time.sleep(4)


NameError: name 'client' is not defined

In [7]:
import time
# --- B. Google ADK (minimal agent) ---
import asyncio
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

adk_agent = Agent(
    model=MODEL,
    name="week1_demo_agent",
    instruction="Answer concisely, in 1-2 sentences.",
)

def ask_adk_agent(agent, prompt, app_name="week1_app", user_id="student"):
    """Small reusable helper: runs one prompt through an ADK agent and returns the final text."""
    runner = InMemoryRunner(agent=agent, app_name=app_name)
    session = asyncio.run(runner.session_service.create_session(app_name=app_name, user_id=user_id))
    content = types.Content(role="user", parts=[types.Part.from_text(text=prompt)])
    final_text = None
    for event in runner.run(user_id=user_id, session_id=session.id, new_message=content):
        if event.content and event.content.parts and event.content.parts[0].text:
            final_text = event.content.parts[0].text
    return final_text

print("B) Google ADK agent:\n", ask_adk_agent(adk_agent, prompt))
time.sleep(4)


NameError: name 'MODEL' is not defined

In [8]:
import time
# --- C. LangChain chat model ---
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model=MODEL)
lc_response = llm.invoke(prompt)
print("C) LangChain:\n", lc_response.content)
time.sleep(4)


NameError: name 'MODEL' is not defined

## 5. Exercises

Complete both before the peer code review activity.

In [9]:
# TODO Exercise 1 — Your own parameter sweep
import time
print('=== EXERCISE 1 RESULT ===')
MY_PROMPT = "Write a short poem about a robot learning to love."

for temp in [0.0, 0.4, 0.9, 1.5]:
    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=MY_PROMPT,
            config=types.GenerateContentConfig(temperature=temp, max_output_tokens=50),
        )
        text = response.text.strip() if response.text else 'NO TEXT GENERATED'
        print(f"Temperature={temp:<4} -> {text}\n")
    except Exception as e:
        print(f"Temperature={temp:<4} -> ERROR: {e}\n")
    time.sleep(3)


=== EXERCISE 1 RESULT ===
Temperature=0.0  -> ERROR: name 'client' is not defined



Temperature=0.4  -> ERROR: name 'client' is not defined



Temperature=0.9  -> ERROR: name 'client' is not defined



Temperature=1.5  -> ERROR: name 'client' is not defined



In [10]:
# TODO Exercise 2 — Estimate tokens without calling the API
import time
print('\n=== EXERCISE 2 RESULT ===')
def estimate_tokens(text: str) -> float:
    return len(text) / 4.0

for text in samples:
    try:
        real = client.models.count_tokens(model=MODEL, contents=text).total_tokens
        estimate = estimate_tokens(text)
        error_pct = abs(real - estimate) / real * 100 if real > 0 else 0
        print(f"Real: {real:4} | Est: {estimate:6.2f} | Error: {error_pct:5.2f}% | Text: {text[:40]!r}...")
    except Exception as e:
        print(f"Text: {text[:40]!r}... -> ERROR: {e}")
    time.sleep(2)



=== EXERCISE 2 RESULT ===
Text: 'Hi!'... -> ERROR: name 'client' is not defined


Text: 'The capital of France is Paris.'... -> ERROR: name 'client' is not defined


Text: 'AI Agentic Engineering is a course about'... -> ERROR: name 'client' is not defined


## Next week

Week 2 — **Context Engineering I**: system prompts, few-shot prompting, chain-of-thought, and forcing structured
JSON output. See `week-02-context-engineering-i-content.html`.